# CapstoneBO+ Method Demo

This notebook is a compact walkthrough of the code and results pipeline. It shows how the CSV histories are loaded, how progress is summarised, and how the 2D surface plots are produced for the low-dimensional functions. It doesn't show the complete pipeline.

## 1. Set up paths and imports

The path logic below lets the notebook run either from the `CapstoneBO+` folder or from the parent capstone folder.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display

# Find the project folder no matter where Jupyter was launched from.
cwd = Path.cwd()
if (cwd / "data").exists() and (cwd / "plot_function_progress.py").exists():
    PROJECT_ROOT = cwd
elif (cwd / "CapstoneBO+" / "data").exists():
    PROJECT_ROOT = cwd / "CapstoneBO+"
else:
    raise FileNotFoundError("Could not find the CapstoneBO+ project folder.")

DATA_DIR = PROJECT_ROOT / "data"
IMAGES_DIR = PROJECT_ROOT / "images"
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Data folder:   {DATA_DIR}")

## 2. Load the observation histories

Each `function_*.csv` file is a cumulative history of submitted points and returned `y0` values. The `index` column is the submission order.

In [ ]:
def function_sort_key(path: Path) -> tuple[int, str]:
    """Sort function_1, function_2, ... in numeric order."""
    try:
        return int(path.stem.split("_")[-1]), path.stem
    except ValueError:
        return 10_000, path.stem


csv_paths = sorted(DATA_DIR.glob("function_*.csv"), key=function_sort_key)
frames = []
for path in csv_paths:
    df = pd.read_csv(path)
    df["source_file"] = path.name
    frames.append(df)

df_all = pd.concat(frames, ignore_index=True)
display(df_all[["source_file", "function", "index", "x0", "x1", "y0"]].head(12))

## 3. Validate the data labels

Several plotting functions filter rows by the `function` column. This quick check catches rows whose label does not match their filename, which is the usual reason a newly added point does not appear in a plot.

In [ ]:
mismatches = []
for path in csv_paths:
    expected = path.stem
    df = pd.read_csv(path)
    bad = df[df["function"] != expected].copy()
    if not bad.empty:
        bad.insert(0, "expected_function", expected)
        bad.insert(0, "source_file", path.name)
        mismatches.append(bad)

if mismatches:
    mismatch_df = pd.concat(mismatches, ignore_index=True)
    display(mismatch_df[["source_file", "expected_function", "function", "index", "x0", "x1", "y0"]])
else:
    print("All function labels match their filenames.")

## 4. Summarize best-so-far progress

The optimization goal is to maximize `y0`. For each function, the table below records the best observed value and how many submissions happened after that best point.

In [ ]:
from plot_function_progress import build_progress_table

progress_summary = build_progress_table(DATA_DIR)
display(progress_summary)

## 5. Generate and display the progress chart

The helper below writes two artifacts: a PNG chart and a CSV summary. The chart shows the observed `y0` values, the cumulative best-so-far line, the highest `y0` point, and shaded submissions after that highest point.

In [ ]:
from plot_function_progress import plot_function_progress

progress_png = IMAGES_DIR / "function_progress_over_time.png"
progress_csv = IMAGES_DIR / "function_progress_over_time_summary.csv"

plot_function_progress(DATA_DIR, progress_png, progress_csv)
display(Image(filename=str(progress_png)))

## 6. Generate the 2D surface plots

`plot_function_surfaces.py` is designed for `function_1` and `function_2`, because those histories use `x0` and `x1`. It creates a three-panel figure: raw 3D scatter, contour view, and fitted RBF surface.

In [ ]:
from plot_function_surfaces import _load_function, plot_triptych

for function_name in ["function_1", "function_2"]:
    csv_path = DATA_DIR / f"{function_name}.csv"
    out_path = IMAGES_DIR / f"{function_name}_surface_triptych.png"
    sub = _load_function(csv_path, function_name)
    plot_triptych(sub, function_name, out_path)
    print(f"{function_name}: plotted {len(sub)} usable rows -> {out_path.name}")

Display the generated surface images:

In [ ]:
for image_name in ["function_1_surface_triptych.png", "function_2_surface_triptych.png"]:
    display(Image(filename=str(IMAGES_DIR / image_name)))

## 7. Synthetic weekly dashboard preview

The next cells create a tiny dashboard preview using synthetic sample values. The values are illustrative only; they are not model outputs or real recommendations.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
fake_functions = ["function_1", "function_2", "function_3"]

fake_recommendations = pd.DataFrame(
    {
        "function": fake_functions,
        "recommendation_type": ["chosen_submission"] * len(fake_functions),
        "candidate_label": [f"sample_candidate_{i:03d}" for i in range(len(fake_functions))],
        "portal_string": [
            "0.123456-0.654321",
            "0.224466-0.771103",
            "0.558801-0.492210-0.337700",
        ],
        "mu_t": rng.normal(0.5, 0.08, len(fake_functions)).round(4),
        "sigma_t": rng.uniform(0.02, 0.14, len(fake_functions)).round(4),
        "candidate_origin": ["sample_global", "sample_trust_region", "sample_elite"],
        "stagnation_flag": [False, True, False],
    }
)

fake_candidate_rows = []
for fn in fake_functions:
    for rank in range(4):
        fake_candidate_rows.append(
            {
                "function": fn,
                "candidate_label": f"sample_{fn}_candidate_{rank}",
                "is_submission": rank == 0,
                "portal_string": f"{rng.random():.6f}-{rng.random():.6f}",
                "mu_t": round(float(rng.normal(0.45, 0.10)), 4),
                "sigma_t": round(float(rng.uniform(0.02, 0.16)), 4),
                "ei": round(float(rng.uniform(0.0001, 0.0200)), 5),
                "pi": round(float(rng.uniform(0.01, 0.80)), 4),
                "ucb": round(float(rng.normal(0.70, 0.08)), 4),
                "is_pareto": rank < 2,
                "pareto_rank": rank + 1,
                "novelty_nn_dist": round(float(rng.uniform(0.03, 0.40)), 4),
                "boundary_prox": round(float(rng.uniform(0.00, 0.20)), 4),
                "duplicate_after_rounding": False,
            }
        )

fake_candidates = pd.DataFrame(fake_candidate_rows)

print("Synthetic sample values only; not model outputs.")
display(fake_recommendations)
display(fake_candidates.head(8))

In [ ]:
from IPython.display import HTML
from reporting.dashboard import make_weekly_dashboard

fake_dashboard_path = IMAGES_DIR / "fake_weekly_dashboard_preview.html"
make_weekly_dashboard(fake_candidates, fake_recommendations, fake_dashboard_path)

html = fake_dashboard_path.read_text(encoding="utf-8")
sample_note = """
<div class='card' style='border-left:4px solid #64748b;background:#f8fafc'>
  <strong>Synthetic preview:</strong> values below are illustrative only, not model outputs.
</div>
"""
html = html.replace("<body>", f"<body>{sample_note}", 1)
fake_dashboard_path.write_text(html, encoding="utf-8")

print(f"Wrote synthetic dashboard preview to: {fake_dashboard_path}")
display(HTML(html))

## 8. What to run for weekly recommendations

The notebook above focuses on readable demonstration and diagnostics. The full recommendation rebuild is handled by the command-line pipeline:

```bash
python3 weekly_pack.py --report_date YYYY-MM-DD
```

That script fits the surrogate models, generates candidate pools, scores candidates, removes duplicates, and writes report artifacts for review.